# Task 2 — Data Engineering
## Distributed Flight Cancellation Risk Prediction Using PySpark
 
### 1. Data Cleaning and Preprocessing Explanation
Our data cleaning pipeline must address several critical steps:
1. **Removing Duplicates**: We apply `dropDuplicates()` to ensure no flight record is counted twice.
2. **Removing Invalid Records**: Filter to make sure `Cancelled` label column contains only valid values (non-null).
3. **Target Variable Leakage**: Columns like `DepTime`, `DepDelay`, `ArrTime`, and `ArrDelay` are only known *after* a flight operates. If a flight is cancelled, they are missing. Using them will lead to data leakage, so they must be removed.
4. **Imputing Missing Values**: We found that `TaxiOut` is null for cancelled flights (because the plane never taxied). Since the readme requires us to use `TaxiOut` as a numerical feature, we impute these nulls with `0.0` rather than dropping the rows. If we dropped them, we would lose **99.3%** of our cancelled flights!



In [ ]:
import os
import sys
import ctypes

def get_short_path(long_name):
    try:
        buf = ctypes.create_unicode_buffer(1024)
        ctypes.windll.kernel32.GetShortPathNameW(long_name, buf, 1024)
        return buf.value if buf.value else long_name
    except Exception:
        return long_name

# Workaround for Java 18+ Security Manager issue on Windows
os.environ['SPARK_SUBMIT_OPTS'] = '-Djava.security.manager=allow'
os.environ['HADOOP_HOME'] = get_short_path(r'C:\hadoop')

python_exe = get_short_path(sys.executable)
os.environ['PYSPARK_PYTHON'] = python_exe
os.environ['PYSPARK_DRIVER_PYTHON'] = python_exe

java_home = os.environ.get('JAVA_HOME', '')
if not java_home or not os.path.exists(os.path.join(java_home, 'bin', 'java.exe')):
    default_java = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot'
    if os.path.exists(os.path.join(default_java, 'bin', 'java.exe')):
        os.environ['JAVA_HOME'] = get_short_path(default_java)
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("Task2_Data_Engineering") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()

# Load
csv_path = "../../PR/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv"
df = spark.read.csv(csv_path, header=True, inferSchema=True)
print("Data loaded. Raw row count:", df.count())



### 2. Execution of Preprocessing


In [ ]:
# Drop duplicates
clean_df = df.dropDuplicates()

# Filter to keep only operationally relevant rows
clean_df = clean_df.filter(clean_df.Cancelled.isNotNull())

# Impute TaxiOut missing values to 0.0 (since cancelled flights don't taxi)
clean_df = clean_df.fillna({"TaxiOut": 0.0})

# Ensure key columns have no nulls
key_cols = ["Month", "DayOfWeek", "CRSDepTime", "Distance", "Reporting_Airline", "Origin", "Dest"]
clean_df = clean_df.na.drop(subset=key_cols)

print("Duplicates, invalid records, and nulls resolved. Cleaned row count:", clean_df.count())



### 3. Missing Value Summary & Final Schema Preview


In [ ]:
# Display null count of key columns
clean_df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in key_cols + ["TaxiOut"]]).show()



### 4. PySpark ML Pipeline Setup
We build a reusable ML pipeline containing:
- **StringIndexer**: Index the categorical fields (`Reporting_Airline`, `Origin`, `Dest`).
- **OneHotEncoder**: Convert indexed categories into sparse vectors.
- **VectorAssembler**: Merge numerical features (`Month`, `DayOfWeek`, `CRSDepTime`, `Distance`, `TaxiOut`) and OneHotEncoded categories.
- **StandardScaler**: Scale the assembled features to zero mean and unit variance.



In [ ]:
categorical_features = ["Reporting_Airline", "Origin", "Dest"]
numeric_features = ["Month", "DayOfWeek", "CRSDepTime", "Distance", "TaxiOut"]

# 1. String Indexer
indexers = [StringIndexer(inputCol=col, outputCol=col+"_Index", handleInvalid="keep") for col in categorical_features]

# 2. One Hot Encoder
encoder = OneHotEncoder(
    inputCols=[col+"_Index" for col in categorical_features],
    outputCols=[col+"_OHE" for col in categorical_features]
)

# 3. Vector Assembler
assembler = VectorAssembler(
    inputCols=numeric_features + [col+"_OHE" for col in categorical_features],
    outputCol="features"
)

# 4. Standard Scaler
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

# Assemble pipeline (without model stage for now)
pipeline_stages = indexers + [encoder, assembler, scaler]
preprocessing_pipeline = Pipeline(stages=pipeline_stages)

# Fit and transform
pipeline_model = preprocessing_pipeline.fit(clean_df)
processed_df = pipeline_model.transform(clean_df)

print("Pipeline executed successfully!")
processed_df.select("scaledFeatures").show(5, truncate=False)



### 5. Feature Engineering Summary Table
| Feature Name | Type | Processing | Reason / Notes |
|---|---|---|---|
| **Month** | Numeric | Scaled | Temporal indicator for seasonality (Jan). |
| **DayOfWeek** | Numeric | Scaled | Captures day of the week traffic fluctuations. |
| **CRSDepTime** | Numeric | Scaled | Scheduled departure time represents rush hours. |
| **Distance** | Numeric | Scaled | Captures flight distance (longer flights less likely to cancel). |
| **TaxiOut** | Numeric | Imputed (0.0), Scaled | Leakage risk, but represents operations. Imputed to avoid dropping cancelled flights. |
| **Reporting_Airline**| Categorical | StringIndexed, OHE | Captures airline-specific cancellation rates. |
| **Origin** | Categorical | StringIndexed, OHE | Captures departure airport-specific congestion. |
| **Dest** | Categorical | StringIndexed, OHE | Captures arrival airport-specific congestion. |



In [ ]:
spark.stop()
